# YOLO × VisDrone → OpenMV NPU — End-to-End Notebook

This notebook runs the **full pipeline** from this repository in a single place:

1. **Tune** — Ray Tune hyperparameter search (`tune_hyperparameters.py`)
2. **Train** — YOLO11/YOLO26 on VisDrone (`train_and_export.py`)
3. **Export** — INT8 TFLite + Arm Vela (AE3) and STEdgeAI (N6) NPU compilation

It is written to run **locally** on a GPU host **and** on common **cloud GPU notebooks**:

| Environment | Auto-detected | Notes |
|---|---|---|
| Local (`uv sync`-managed) | ✅ | Re-uses the repo's own `.venv/` via `uv run`. |
| Google Colab | ✅ | Clones the repo, provisions Python 3.14 via `uv`, optional Google Drive mount for persistence. |
| Paperspace Gradient | ✅ | Clones into `/notebooks/`, provisions Python 3.14 via `uv`, persists to the mounted `/storage/`. |
| Kaggle / other | ✅ (fallback) | Treated like a generic cloud runtime. |

> **Cloud caveat — session timeouts.** Free Colab and Paperspace sessions reclaim
> the runtime after a few hours. The defaults below (few iterations, ~10 epochs,
> one small model) produce a complete end-to-end run in well under an hour. Scale
> up by editing the **Configuration** cell once the pipeline works.

> **Python version.** The project pins `requires-python = ">=3.14"`. Cloud
> runtimes still ship with older interpreters, so the install cell below uses
> `uv python install 3.14` + `uv sync` to provision the right CPython build
> and every pipeline step is driven through that venv.

> **NPU compilation.** Arm Vela (AE3) is `pip`-installable and runs automatically.
> STEdgeAI (N6, proprietary) is skipped unless you mount / install it — the raw
> INT8 TFLite produced here still loads directly on the OpenMV N6 firmware.

## 1. Environment detection

Figures out whether we're running locally, in Colab, or in Paperspace Gradient, and
picks sensible repo / dataset / output roots for each.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

def detect_env() -> str:
    if "google.colab" in sys.modules or os.path.isdir("/content"):
        return "colab"
    if os.environ.get("PAPERSPACE_CLUSTER_ID") or os.path.isdir("/notebooks"):
        return "paperspace"
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.path.isdir("/kaggle"):
        return "kaggle"
    return "local"

ENV = detect_env()
REPO_URL = "https://github.com/TankMasterRL/yolo11-26-visdrone-openmv-npu.git"
REPO_NAME = "yolo11-26-visdrone-openmv-npu"

if ENV == "colab":
    WORKSPACE = Path("/content")
elif ENV == "paperspace":
    WORKSPACE = Path("/notebooks")
elif ENV == "kaggle":
    WORKSPACE = Path("/kaggle/working")
else:
    # Local: assume the notebook lives under <repo>/notebooks/
    WORKSPACE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

REPO_DIR = WORKSPACE / REPO_NAME if ENV != "local" else WORKSPACE

print(f"Environment : {ENV}")
print(f"Workspace   : {WORKSPACE}")
print(f"Repo dir    : {REPO_DIR}")

## 2. Clone the repository (cloud only)

On cloud runtimes the filesystem is ephemeral, so we clone the project on each
session start. Locally this is a no-op.

In [ ]:
if ENV != "local":
    if not (REPO_DIR / "pyproject.toml").exists():
        print(f"Cloning {REPO_URL} → {REPO_DIR}")
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
            check=True,
        )
    else:
        print(f"Repo already present at {REPO_DIR}")
else:
    assert (REPO_DIR / "pyproject.toml").exists(), (
        f"Could not find pyproject.toml in {REPO_DIR}. Run the notebook from\n"
        f"inside the cloned repo (or its notebooks/ subdirectory)."
    )

os.chdir(REPO_DIR)
print(f"Working dir : {Path.cwd()}")

## 3. Install dependencies (Python 3.14 via uv)

The repo's `pyproject.toml` pins `requires-python = ">=3.14"`, but Colab and
Paperspace runtimes ship older system Pythons (3.10 / 3.11). Instead of
fighting the kernel's interpreter, we let [`uv`](https://docs.astral.sh/uv/)
download the required CPython build and provision a project-managed `.venv/`
— the same `uv sync --extra all` workflow the README and Dockerfile use.

From here on **every** script invocation goes through `uv run` so it picks
up that Python 3.14 venv rather than the notebook kernel's older interpreter.

In [ ]:
def run(cmd, **kw):
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, check=True, **kw)

# 1) Install the uv binary itself (tiny, no deps) into the kernel's Python.
if shutil.which("uv") is None:
    run([sys.executable, "-m", "pip", "install", "--quiet", "uv"])

# 2) Ask uv to download CPython 3.14 if it isn't already on the machine.
#    On Colab / Paperspace this is a one-off ~30 MB download; locally it is a
#    no-op if pyenv / uv already has 3.14.
run(["uv", "python", "install", "3.14"])

# 3) Create / update the project venv with all extras. This is the notebook
#    equivalent of the README's `uv sync --extra all` command — one shared
#    source of truth for deps across host, Docker, and cloud notebook runs.
os.environ["UV_LINK_MODE"] = "copy"  # /tmp and .venv may be on separate mounts
run(["uv", "sync", "--python", "3.14", "--extra", "all"])

# 4) Record the venv Python so later cells can drive every script through it,
#    independent of whatever interpreter the Jupyter kernel is running.
PYTHON_BIN = str(REPO_DIR / ".venv" / "bin" / "python")
assert Path(PYTHON_BIN).exists(), f"Expected uv venv at {PYTHON_BIN}"
print(f"\nDependency install complete.\n  venv python: {PYTHON_BIN}")

## 4. GPU / runtime check

Fail fast if no CUDA device is visible — training and tuning on CPU are
impractical for VisDrone. On Colab, enable **Runtime → Change runtime type →
GPU** before running this cell. On Paperspace Gradient, start the notebook on a
**GPU** instance type.

In [ ]:
# Query the *venv* interpreter (not the notebook kernel) so we are checking
# exactly the environment the pipeline scripts will use.
import json as _json, textwrap

probe = textwrap.dedent("""
    import json, sys
    try:
        import torch
        cuda = torch.cuda.is_available()
        devices = []
        if cuda:
            for i in range(torch.cuda.device_count()):
                p = torch.cuda.get_device_properties(i)
                devices.append([i, torch.cuda.get_device_name(i), p.total_memory / 1024**3])
        torch_ver = torch.__version__
    except Exception as e:
        cuda, devices, torch_ver = False, [], f"<import failed: {e}>"
    try:
        import ultralytics; ultra_ver = ultralytics.__version__
    except Exception as e: ultra_ver = f"<{e}>"
    try:
        import ray; ray_ver = ray.__version__
    except Exception as e: ray_ver = f"<{e}>"
    print(json.dumps({
        "python":  sys.version.split()[0],
        "torch":   torch_ver,
        "cuda":    cuda,
        "devices": devices,
        "ultralytics": ultra_ver,
        "ray":     ray_ver,
    }))
""")

out = subprocess.check_output([PYTHON_BIN, "-c", probe], text=True)
info = _json.loads(out.strip().splitlines()[-1])

print(f"Python        : {info['python']}")
print(f"PyTorch       : {info['torch']}")
print(f"CUDA available: {info['cuda']}")
for i, name, total in info["devices"]:
    print(f"  [{i}] {name}  ({total:.1f} GB)")
print(f"Ultralytics   : {info['ultralytics']}")
print(f"Ray           : {info['ray']}")

if not info["cuda"]:
    print(
        "\nWARNING: no CUDA GPU detected. Training will be extremely slow.\n"
        "  Colab      → Runtime → Change runtime type → GPU\n"
        "  Paperspace → start the notebook on a GPU instance type"
    )

## 5. Persistent dataset + output directories

`train_and_export.py` honours the `YOLO_DATASETS_DIR` environment variable (the
same one used by the Docker Compose services) so the ~2 GB VisDrone download can
land on a persistent volume:

- **Colab**: optionally mount Google Drive and write to `MyDrive/visdrone-cache/`.
- **Paperspace**: use the built-in persistent `/storage/` volume.
- **Kaggle / local**: fall back to an in-repo `datasets/` directory.

Flip `USE_GOOGLE_DRIVE = True` on Colab if you want dataset + trained weights to
survive runtime recycles.

In [ ]:
USE_GOOGLE_DRIVE = False  # set True on Colab to persist across sessions

if ENV == "colab" and USE_GOOGLE_DRIVE:
    from google.colab import drive  # type: ignore[import-not-found]
    drive.mount("/content/drive")
    PERSIST_ROOT = Path("/content/drive/MyDrive/visdrone-cache")
elif ENV == "paperspace" and Path("/storage").exists():
    PERSIST_ROOT = Path("/storage/visdrone-cache")
else:
    PERSIST_ROOT = REPO_DIR / ".cache"

DATASETS_DIR = PERSIST_ROOT / "datasets"
RUNS_DIR     = REPO_DIR / "runs"         # TB logs stay in-repo so TB works out of the box
EXPORT_DIR   = REPO_DIR / "export"

DATASETS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["YOLO_DATASETS_DIR"] = str(DATASETS_DIR)

print(f"Datasets : {DATASETS_DIR}")
print(f"Runs     : {RUNS_DIR}")
print(f"Exports  : {EXPORT_DIR}")

## 6. Configuration

One cell to tweak before running the pipeline. The defaults are **deliberately
small** so a full tune → train → export run finishes comfortably inside a free
Colab / Paperspace session. Scale `EPOCHS_*`, `TUNE_ITERATIONS`, and `MODELS` up
for production runs (or switch to the Docker Compose setup on a dedicated box).

In [ ]:
# Which models to run through the pipeline. Pick one or more of:
#   yolo11n, yolo11s, yolo26n, yolo26s
MODELS = ["yolo11n"]

# -- Tune --------------------------------------------------------------
TUNE_ITERATIONS = 4    # Ray Tune trials per model (README default: 10)
TUNE_EPOCHS     = 10   # max epochs per trial (README default: 30)
TUNE_GRACE      = 3    # ASHA grace period (README default: 10)
TUNE_IMGSZ      = 640  # image size during tuning

# -- Train -------------------------------------------------------------
TRAIN_EPOCHS = 20      # README default: 100
TRAIN_IMGSZ  = 640

# -- Export ------------------------------------------------------------
# None → script picks 256 for nano, 320 for small (matches README)
EXPORT_IMGSZ = None
SKIP_NPU     = False   # set True to skip Vela / STEdgeAI compilation

print("Pipeline configuration:")
print(f"  models            = {MODELS}")
print(f"  tune iterations   = {TUNE_ITERATIONS}")
print(f"  tune epochs       = {TUNE_EPOCHS} (grace={TUNE_GRACE})")
print(f"  train epochs      = {TRAIN_EPOCHS}")
print(f"  train imgsz       = {TRAIN_IMGSZ}")
print(f"  export imgsz      = {EXPORT_IMGSZ or 'auto (256/320)'}")
print(f"  skip NPU compile  = {SKIP_NPU}")

## 7. Step 1 — Hyperparameter tuning (Ray Tune + TensorBoard)

Runs `tune_hyperparameters.py` with the VisDrone-focused search space. Each
model gets its own Ray Tune experiment directory under `runs/tune/` and a
`*_best_hyperparameters.json` summary file.

We stream the script's stdout into the notebook so progress is visible even in
long cloud sessions.

In [ ]:
def stream(cmd: list[str]) -> int:
    """Run a subprocess and stream its output live into the notebook."""
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, text=True,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    return proc.wait()

tune_cmd = [
    PYTHON_BIN, "tune_hyperparameters.py",
    "--models",        *MODELS,
    "--iterations",    str(TUNE_ITERATIONS),
    "--epochs",        str(TUNE_EPOCHS),
    "--grace-period",  str(TUNE_GRACE),
    "--imgsz",         str(TUNE_IMGSZ),
    "--output",        str(RUNS_DIR / "tune"),
]
rc = stream(tune_cmd)
assert rc == 0, f"tune_hyperparameters.py exited with status {rc}"

# Surface each model's best-hyperparameters file
import json
best_cfgs: dict[str, dict] = {}
for model in MODELS:
    cfg_path = RUNS_DIR / "tune" / f"visdrone_raytune_{model}_best_hyperparameters.json"
    if cfg_path.exists():
        best_cfgs[model] = json.loads(cfg_path.read_text())
        print(f"\nBest hyperparameters for {model}:")
        print(json.dumps(best_cfgs[model], indent=2, default=str))
    else:
        print(f"\n[!] No best-hyperparameters file for {model} at {cfg_path}")

### Optional — launch TensorBoard

Ultralytics writes per-trial TFEvent logs into `runs/tune/...`. Uncomment the
right line for your environment to inspect metrics while the pipeline runs.

In [ ]:
# --- Colab / Jupyter inline TensorBoard -------------------------------
# %load_ext tensorboard
# %tensorboard --logdir runs/tune

# --- Local or Paperspace, standalone TensorBoard server ---------------
# !tensorboard --logdir runs/tune --host 0.0.0.0 --port 6006 &

print("TensorBoard cell is intentionally inert — uncomment one option above.")

## 8. Step 2 — Training

Trains each selected model on VisDrone by running `train_and_export.py` with
`--skip-npu`. We deliberately split training and export into two cells so the
notebook mirrors the three README stages exactly — NPU compilation happens in
the next cell alongside the TFLite export.

> The best-hyperparameters JSON files from Step 1 are **not** automatically
> fed back into training — that is a deliberate README design choice (*"Copy
> the values ... into `DEFAULT_TRAIN_ARGS` of `train_and_export.py`"*). If you
> want to apply them here, edit `train_and_export.py` between the two cells,
> or pass them as `model.train()` overrides.

In [ ]:
train_cmd = [
    PYTHON_BIN, "train_and_export.py",
    "--models",  *MODELS,
    "--epochs",  str(TRAIN_EPOCHS),
    "--imgsz",   str(TRAIN_IMGSZ),
    "--project", str(RUNS_DIR / "visdrone"),
    "--output",  str(EXPORT_DIR),
    "--skip-npu",  # NPU compilation happens in the next cell
]
rc = stream(train_cmd)
assert rc == 0, f"train_and_export.py (train phase) exited with status {rc}"

# Sanity-check that best.pt exists for each model
for model in MODELS:
    best_pt = RUNS_DIR / "visdrone" / model / "weights" / "best.pt"
    status  = "✓" if best_pt.exists() else "✗"
    size_mb = best_pt.stat().st_size / 1024**2 if best_pt.exists() else 0
    print(f"  {status} {model:8s}  {best_pt}  ({size_mb:.1f} MB)")

## 9. Step 3 — Export to INT8 TFLite + NPU compilation

Re-runs `train_and_export.py --skip-train`, which:

1. Loads each `best.pt` produced in the training cell.
2. Exports to **INT8 TFLite** at 256 (nano) or 320 (small) by default.
3. Compiles each TFLite with **Arm Vela** for the AE3 (Ethos-U55).
4. Compiles each TFLite with **STEdgeAI** for the N6 (Neural-ART) — only if
   `stedgeai` is on `PATH`. Without it the script prints a warning and the
   raw INT8 TFLite is still written to `export/n6/<model>/`.

In [ ]:
export_cmd = [
    PYTHON_BIN, "train_and_export.py",
    "--skip-train",
    "--models",  *MODELS,
    "--project", str(RUNS_DIR / "visdrone"),
    "--output",  str(EXPORT_DIR),
]
if EXPORT_IMGSZ is not None:
    export_cmd += ["--imgsz-export", str(EXPORT_IMGSZ)]
if SKIP_NPU:
    export_cmd += ["--skip-npu"]

rc = stream(export_cmd)
assert rc == 0, f"train_and_export.py (export phase) exited with status {rc}"

## 10. Inspect the exported artefacts

Mirrors the tree diagram in the README's *Output structure* section so you can
verify every model, board, and fallback file is in place.

In [ ]:
def tree(root: Path, prefix: str = "") -> None:
    entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    for i, entry in enumerate(entries):
        connector = "└── " if i == len(entries) - 1 else "├── "
        if entry.is_dir():
            print(f"{prefix}{connector}{entry.name}/")
            tree(entry, prefix + ("    " if i == len(entries) - 1 else "│   "))
        else:
            size_kb = entry.stat().st_size / 1024
            print(f"{prefix}{connector}{entry.name}  ({size_kb:.0f} KB)")

print(f"{EXPORT_DIR}/")
tree(EXPORT_DIR)

## 11. Save results off the ephemeral runtime (cloud only)

Cloud sessions lose their local disk on shutdown. Copy the `export/` tree to a
persistent location so you don't have to re-train after a timeout.

- **Colab**: copied to Google Drive when `USE_GOOGLE_DRIVE = True` in Step 5.
- **Paperspace**: copied to `/storage/` when available.
- **Local / Kaggle**: no-op (local disk is already persistent).

In [ ]:
if ENV in ("colab", "paperspace") and PERSIST_ROOT != REPO_DIR / ".cache":
    dst = PERSIST_ROOT / "export"
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(EXPORT_DIR, dst)
    print(f"Copied {EXPORT_DIR} → {dst}")
else:
    print(f"Persisted in place at {EXPORT_DIR} (ENV={ENV})")

## 12. Next steps — deploying to OpenMV

Follow the README's **Deploying to OpenMV Cameras** section:

**OpenMV AE3**

1. Copy `export/ae3/<model>/<model>_int8_vela.tflite` + `export/labels.txt` to
   the camera's filesystem.
2. Copy `openmv-scripts/region_counter.py` and rename
   `openmv-scripts/ae3/main_<model>.py` → `main.py`.
3. Reset the board.

**OpenMV N6**

1. Copy `export/n6/<model>/<model>_int8.tflite` + `export/labels.txt`.
2. Copy `openmv-scripts/region_counter.py` and rename
   `openmv-scripts/n6/main_<model>.py` → `main.py`.
3. Reset the board.

If this notebook was run on a cloud GPU, use the OpenMV IDE on your local
machine to upload the files downloaded from Drive / `/storage/`.

Done — full pipeline complete.